In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
import numpy as np
import znnl as nl

import pandas as pd

from flax import linen as nn
import optax

import matplotlib.pyplot as plt
from neural_tangents import stax

import h5py as hf
from scipy.stats import pearsonr

from rich.progress import track

import seaborn as sns

## Download the data

In [ ]:
generator = nl.data.MNISTGenerator(ds_size=1000)

In [ ]:
ensembles = 20
optimizers = ["sgd", "traceopt", "adam"]

for i in range(ensembles):
    
    generator = nl.data.MNISTGenerator(ds_size=5000)
    seed = np.random.randint(low=0, high=564864168)
    for opt in optimizers:
        network = stax.serial(
          stax.Conv(32, filter_shape=(2, 2)),
          stax.Relu(),
          stax.AvgPool(window_shape=(2, 2), strides=(2, 2)),
          stax.Flatten(),
          stax.Dense(128),
          stax.Relu(),
          stax.Dense(10)
        )
        
        if opt == "traceopt":
            optimizer = nl.optimizers.TraceOptimizer(
                scale_factor=1.0, subset=0.2, memory=100
            )
        elif opt == "adam":
            optimizer = optax.adam(1e-3)
        elif opt == "sgd":
            optimizer = optax.sgd(1.0)
        else:
            raise NotImplementedError(
                "Optimizer doesn't exist..."
            )

        model = nl.models.NTModel(
                nt_module=network,
                optimizer=optimizer,
                seed=seed,
                input_shape=(1, 28, 28, 1),
            )

        train_recorder = nl.training_recording.JaxRecorder(
            name=f"{opt}_train_recorder_{i}",
            loss=True,
            accuracy=True,
            update_rate=1
        )
        test_recorder = nl.training_recording.JaxRecorder(
            name=f"{opt}_test_recorder_{i}",
            loss=True,
            accuracy=True,
            update_rate=1
        )

        train_recorder.instantiate_recorder(
            data_set=generator.train_ds
        )
        test_recorder.instantiate_recorder(
            data_set=generator.test_ds
        )

        training_strategy = nl.training_strategies.SimpleTraining(
            model=model, 
            loss_fn=nl.loss_functions.CrossEntropyLoss(),
            accuracy_fn=nl.accuracy_functions.LabelAccuracy(),
            recorders=[train_recorder, test_recorder],
        )
        _ = training_strategy.train_model(
                train_ds=generator.train_ds,
                test_ds=generator.test_ds, 
                epochs=50, 
                batch_size=128,
            )

        train_recorder.dump_records()
        test_recorder.dump_records()

In [ ]:
def load_data(file):
    with hf.File(file, "r") as db:
        data = db["loss"][:]
        
    return data

adam_data = []

for i in range(10):
    adam_data.append(
        load_data(f"adam_test_recorder_{i}.h5")
    )
    
traceopt_data = []

for i in range(10):
    traceopt_data.append(
        load_data(f"traceopt_test_recorder_{i}.h5")
    )


x = np.linspace(1, 51, 50)

plt.errorbar(
    x,
    np.mean(traceopt_data, axis=0), 
    yerr=np.std(traceopt_data, axis=0),
    marker='x',
    c = "#140D4F",
    mfc="none",
    linestyle="none",
    label="TraceOpt"
)
plt.errorbar(
    x,
    np.mean(adam_data, axis=0), 
    yerr=np.std(adam_data, axis=0),
    marker='o',
    c="#4EA699",
    mfc="none",
    linestyle="none",
    label="ADAM"
)

plt.xlabel("Epoch")
plt.ylabel(r"$\mathcal{L}_{test}$")
plt.yscale("log")
plt.xscale("log")
plt.legend()
plt.savefig("mpg-vs-adam.pdf")
plt.show()